# Creating a train-val-test split
Treating each page as a data point.
Ration used is 60:20:20.

Sample Size = 400 pages  
Train = 240 pages  
Val = 80 pages  
Test = 80 pages  

Steps:
1. Keep selecting one page from random PDFs till the total page count reaches the desired value. Repeat for train, val and test.
2. Separate folders are created for train, val and test. Each of them contains folder for images and annotations.
3. The annotations for val and test will be labelled using Label Studio. See README for info on Label Studio.

In [1]:
%pip install pymupdf==1.24.7

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pymupdf, json, os, random, shutil
from glob import glob

In [3]:
pdfs = glob(r"C:\Users\ashar\repos\ai4doc\pdfs\*.pdf")

In [4]:
# Set the seed for reproducibility
seed = 42
random.seed(seed)

In [5]:
directories = ["train", "val", "test"]
sub = ["images", "annotations"]

# Check if the folder exists
if os.path.exists("dataset"):
    # If the folder is not empty, use shutil.rmtree to delete it
    shutil.rmtree("dataset")

for x in directories:
    for y in sub:
        os.makedirs(f"dataset/{x}/{y}")

In [6]:
def filter_text(json_dict):
    anno_list= []
    id = 0
    for x in json_dict["blocks"]:
        if x['type'] == 0:
            for y in x["lines"]:
                for x in y["spans"]:
                    anno_list+=[{"id":id, "text":x["text"], "box":x["bbox"]}]

    return anno_list
        

In [7]:
def obtain_anno_and_image(pdf, anno_dir, im_dir, page_num):
    print(f"Processing {os.path.basename(pdf).split(".")[0]}")
    # Open the PDF file
    doc = pymupdf.open(pdf)
    page = doc.load_page(page_num)
    
    # Set the zoom matrix
    zoom_factor = 10
    zoom_matrix = pymupdf.Matrix(zoom_factor, zoom_factor)
    
    # Extract text from the page
    text = page.get_text("json")

    # Convert JSON string to dictionary
    json_dict = json.loads(text)
    json_list = filter_text(json_dict)

    # Render page to an image
    pix = page.get_pixmap(matrix=zoom_matrix)

    # Writing json
    fname = f"{os.path.basename(pdf).split(".")[0]}_{page_num}"
    output_json = fname + ".json"
    with open(os.path.join(anno_dir, output_json), "w", encoding="utf-8") as f:
        json.dump(json_list, f, ensure_ascii=False, indent=4)

    # Writing image
    output_image = fname + ".png"
    pix.save(os.path.join(im_dir, output_image))

In [8]:
sample_size = 400
train_pct = 60
val_pct = 20
test_pct = 20

train_size = sample_size*train_pct/100
val_size = sample_size*val_pct/100
test_size = sample_size*test_pct/100

# Generating Data

In [9]:
def gen_sample(count, anno_dir, im_dir):
    for x in range(int(count)):
        selected_pdf = random.sample(pdfs, 1)[0]
        doc = pymupdf.open(selected_pdf)
        num_pages = len(doc)
        selected_page = random.sample(range(num_pages), 1)[0]

        obtain_anno_and_image(
            pdf=selected_pdf, 
            anno_dir=anno_dir, 
            im_dir=im_dir, 
            page_num = selected_page
            )


In [10]:
gen_sample(train_size, anno_dir=r"dataset\train\annotations", im_dir=r"dataset\train\images")
gen_sample(val_size, anno_dir=r"dataset\val\annotations", im_dir=r"dataset\val\images")
gen_sample(test_size, anno_dir=r"dataset\test\annotations", im_dir=r"dataset\test\images")



Processing s71809-69
Processing s70217-1839300-154914
Processing s70718-4180484-172374
Processing s70616-215
Processing s70515-20154066-322242
Processing s71516-30
Processing s70222-20124000-280138
Processing s70423-236440-493922
Processing s70715-20128837-294678
Processing s71411-368
Processing s72219-6743768-207864
Processing s72219-6639897-203557
Processing s70713-645
Processing s71515-2
Processing s72622-20157322-325665
Processing s72410-39
Processing s70623-198899-398742
Processing s70913-265
Processing s70622-20123410-279671
Processing s74510-515
Processing s70515-20144281-309230
Processing s71009-359
Processing s70922-20128313-291071
Processing s71615-62
Processing s72622-20157230-325499
Processing s72311-14
Processing s71315-40
Processing s74510-772
Processing s70322-20161798-330674
Processing s73022-20154553-322736
Processing s71416-33
Processing s72219-6702759-206068
Processing s70313-217
Processing s72415-83156-216891
Processing s70411-33
Processing s73411-141
Processing s71